# Load Libraries

In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch import nn, optim
from torch.optim import lr_scheduler
from torch.cuda.amp import GradScaler, autocast

import numpy as np
import pandas as pd
import os

# Import Pre-trained Image Models (TIMM)
import timm

from util.Trainer import EyeDiseaseClassifier, train_model, evaluate_model

c:\Users\kanek\miniconda3\envs\PyTorch-DL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Global Variables

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 4
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
NUM_CLASSES = 9
IMG_SIZE = 224
NUM_WORKERS = 4
DROP_RATE = 0.5
WEIGHT_DECAY = 1e-4

Using device: cuda:0


# Load Data

In [3]:
#  Load Data from pt file
base_path = "./Data/Final_Dataset/"
train_ds = torch.load(base_path + "train_dataset.pt")
balanced_train_ds = torch.load(base_path + "balanced_train_dataset_dataset.pt")
test_ds = torch.load(base_path + "test_dataset.pt")
valid_ds = torch.load(base_path + "val_dataset.pt")
class_names = torch.load(base_path + "class_map.pt")

# Create DataLoaders

In [4]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
balanced_train_loader = DataLoader(balanced_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [5]:
original_dataloaders = {
    "train": train_loader,
    "valid": valid_loader,
    "test": test_loader
}

balanced_dataloaders = {
    "train": balanced_train_loader,
    "valid": valid_loader,
    "test": test_loader
}

# Create Model

In [6]:
model = EyeDiseaseClassifier(
        model_name="efficientnet_b0",
        num_classes=NUM_CLASSES,
        dropout_rate=DROP_RATE,
        in_channels=1
    )
model = model.to(device)

# Set Training Param

In [7]:
criterion = nn.CrossEntropyLoss()
    
# Define optimizer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Define learning rate scheduler
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# Train Model

## Original Dataset

In [9]:
# Train the model
model, history = train_model(
        model=model,
        dataloaders={'train': train_loader, 'val': valid_loader},
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        num_epochs=NUM_EPOCHS,
    )

Epoch 1/10
----------


train: 100%|██████████| 665/665 [01:31<00:00,  7.23it/s]


train Loss: 1.6117 Acc: 0.4321


val: 100%|██████████| 333/333 [00:51<00:00,  6.50it/s]


val Loss: 1.6565 Acc: 0.4424

Epoch 2/10
----------


...

val Loss: 1.0470 Acc: 0.6606

Epoch 10/10
----------


train: 100%|██████████| 665/665 [01:28<00:00,  7.49it/s]


train Loss: 0.4111 Acc: 0.8432


val: 100%|██████████| 333/333 [00:42<00:00,  7.82it/s]

val Loss: 1.0404 Acc: 0.6659

Training complete in 22m 33s
Best val Acc: 0.6667


In [10]:
results = evaluate_model(
        model=model,
        test_loader=test_loader,
        device=device,
        class_names=class_names
    )

Evaluating: 100%|██████████| 333/333 [00:43<00:00,  7.74it/s]


Classification Report:
                                               precision    recall  f1-score   support

                                      Healthy       0.60      0.14      0.23        21
Central Serous Chorioretinopathy-Color Fundus       0.87      0.86      0.86       386
                         Diabetic Retinopathy       0.61      0.59      0.60        37
                                   Disc Edema       0.65      0.66      0.65       325
                                     Glaucoma       0.66      0.72      0.69       265
                                 Macular Scar       0.55      0.47      0.51       112
                                       Myopia       0.59      0.57      0.58       120
                           Retinal Detachment       0.79      0.79      0.79        29
                         Retinitis Pigmentosa       0.54      0.80      0.64        35

                                     accuracy                           0.70      1330
                 

## Balanced Dataset

In [9]:
# Train the model
model, history = train_model(
        model=model,
        dataloaders={'train': balanced_train_loader, 'val': valid_loader},
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        num_epochs=NUM_EPOCHS,
    )

Epoch 1/10
----------


train:   0%|          | 0/1677 [00:00<?, ?it/s]

train: 100%|██████████| 1677/1677 [03:39<00:00,  7.62it/s]


train Loss: 1.5386 Acc: 0.4438


val: 100%|██████████| 333/333 [00:53<00:00,  6.27it/s]


...

val Loss: 0.9659 Acc: 0.7208

Epoch 10/10
----------


train: 100%|██████████| 1677/1677 [03:19<00:00,  8.41it/s]


train Loss: 0.1395 Acc: 0.9487


val: 100%|██████████| 333/333 [00:46<00:00,  7.17it/s]


val Loss: 0.9650 Acc: 0.7254

Training complete in 41m 45s
Best val Acc: 0.7336


In [10]:
results = evaluate_model(
        model=model,
        test_loader=test_loader,
        device=device,
        class_names=class_names
    )

Evaluating: 100%|██████████| 333/333 [00:45<00:00,  7.34it/s]


Classification Report:
                                               precision    recall  f1-score   support

                                      Healthy       0.80      0.95      0.87        21
Central Serous Chorioretinopathy-Color Fundus       0.93      0.84      0.89       386
                         Diabetic Retinopathy       0.80      0.97      0.88        37
                                   Disc Edema       0.64      0.67      0.65       325
                                     Glaucoma       0.76      0.67      0.71       265
                                 Macular Scar       0.66      0.76      0.71       112
                                       Myopia       0.66      0.66      0.66       120
                           Retinal Detachment       0.63      1.00      0.77        29
                         Retinitis Pigmentosa       0.79      0.94      0.86        35

                                     accuracy                           0.75      1330
                 